# 🚀 Lab 01: Hello World RAG - RAG 基礎入門

## 學習目標
在本實驗中，您將學習：
1. **文件加載 (Document Loading)** - 如何載入不同格式的文件
2. **文本分割 (Text Splitting)** - 將長文件切分為適當大小的 chunks
3. **向量嵌入 (Embedding)** - 使用本地模型將文本轉換為向量
4. **向量存儲 (Vector Store)** - 使用 ChromaDB 存儲和檢索向量
5. **完整 RAG Pipeline** - 整合本地 LLM 建立問答系統

## 技術棧 (全部本地執行)
- **Embedding Model**: `sentence-transformers/all-MiniLM-L6-v2`
- **Vector Database**: ChromaDB
- **LLM**: Ollama (llama3.2 或其他本地模型)
- **Framework**: LangChain

---


In [ ]:
# 確認安裝成功
import langchain
import chromadb
import sentence_transformers

print(f"✅ LangChain version: {langchain.__version__}")
print(f"✅ ChromaDB version: {chromadb.__version__}")
print(f"✅ Sentence Transformers version: {sentence_transformers.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.8 MB/s eta 0:00:00


### 1.2 設置 Ollama (本地 LLM)

在開始之前，請確保您已安裝 Ollama 並下載模型：

```bash
# 安裝 Ollama (請參考 https://ollama.ai)
# Windows: 下載安裝檔
# Mac: brew install ollama
# Linux: curl -fsSL https://ollama.ai/install.sh | sh

# 下載模型 (選擇其一)
ollama pull llama3.2        # 推薦，較新的模型
ollama pull llama3.2:1b     # 輕量版，適合低配置電腦
ollama pull gemma2:2b       # Google 的小型模型
```



In [ ]:
# 測試 Ollama 連接
from langchain_ollama import OllamaLLM

# 設定模型名稱 (根據您下載的模型調整)
MODEL_NAME = "llama3.2"  # 或 "llama3.2:1b", "gemma2:2b"

try:
    llm = OllamaLLM(model=MODEL_NAME)
    response = llm.invoke("Hello! Please respond with 'I am ready!'")
    print(f"✅ Ollama 連接成功！")
    print(f"📝 模型回應: {response}")
except Exception as e:
    print(f"❌ Ollama 連接失敗: {e}")
    print("請確保 Ollama 服務正在運行 (ollama serve)")

Directory 'data' created.


---
## 📄 Part 2: 文件加載 (Document Loading)

RAG 的第一步是將資料載入系統。LangChain 提供多種 Document Loaders 支援不同格式。



In [ ]:
import os
import requests

# 建立資料目錄
os.makedirs('data', exist_ok=True)

# 下載範例文件：愛麗絲夢遊仙境
url = "https://www.gutenberg.org/files/11/11-0.txt"
output_path = "data/alice_in_wonderland.txt"

if not os.path.exists(output_path):
    response = requests.get(url)
    response.raise_for_status()
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"✅ 已下載: {output_path}")
else:
    print(f"📁 檔案已存在: {output_path}")

Successfully downloaded 'data/alice_in_wonderland.txt'.


In [ ]:
from langchain_community.document_loaders import TextLoader

# 使用 TextLoader 載入文件
loader = TextLoader('data/alice_in_wonderland.txt', encoding='utf-8')
documents = loader.load()

print(f"📚 載入了 {len(documents)} 個文件")
print(f"📏 文件長度: {len(documents[0].page_content):,} 字元")
print(f"\n📝 文件前 500 字元預覽:")
print("-" * 50)
print(documents[0].page_content[:500])


---
## ✂️ Part 3: 文本分割 (Text Splitting)

為什麼需要分割文本？
1. **LLM 上下文限制**: 模型有 token 數量限制
2. **檢索精確度**: 較小的 chunks 能提供更精確的檢索結果
3. **語義完整性**: 適當的分割保留文本的語義連貫性

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 建立文本分割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # 每個 chunk 的最大字元數
    chunk_overlap=200,    # chunks 之間的重疊字元數
    length_function=len,
    add_start_index=True, # 記錄每個 chunk 在原文中的位置
)

# 分割文件
chunks = text_splitter.split_documents(documents)

print(f"📄 原始文件數: {len(documents)}")
print(f"📝 分割後 chunks 數: {len(chunks)}")
print(f"📊 平均 chunk 長度: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} 字元")



📄 原始文件數: 1
📝 分割後 chunks 數: 191
📊 平均 chunk 長度: 825 字元


In [ ]:
# 查看分割結果
print("🔍 前 3 個 chunks 預覽:")
print("=" * 60)

for i, chunk in enumerate(chunks[:3]):
    print(f"\n📌 Chunk {i+1}:")
    print(f"   長度: {len(chunk.page_content)} 字元")
    print(f"   Metadata: {chunk.metadata}")
    print(f"   內容預覽: {chunk.page_content[:150]}...")
    print("-" * 60)

: 

---
## 🧮 Part 4: 向量嵌入 (Embedding)

Embedding 是將文本轉換為數值向量的過程，使電腦能夠理解語義相似性。



In [8]:
import os

os.environ["HF_HOME"] = "./hf_cache"   # 或 "./models"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# 使用本地 embedding 模型
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_folder="./hf_cache",
    model_kwargs={"device": "cuda"},
    encode_kwargs={'normalize_embeddings': True}  # 正規化向量
)

print("✅ Embedding 模型載入成功！")

: 

In [ ]:
import numpy as np

# 測試 embedding 和語義相似度
test_texts = [
    "Alice fell down the rabbit hole.",
    "A girl tumbled into a deep burrow.",
    "The weather is sunny today."
]

# 生成 embeddings
test_embeddings = embeddings.embed_documents(test_texts)

print(f"📐 Embedding 維度: {len(test_embeddings[0])}")

# 計算語義相似度 (餘弦相似度)
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print("\n📊 語義相似度分析:")
print("=" * 60)

for i in range(len(test_texts)):
    for j in range(i+1, len(test_texts)):
        sim = cosine_similarity(test_embeddings[i], test_embeddings[j])
        print(f"\n🔹 文本 {i+1} vs 文本 {j+1}:")
        print(f"   '{test_texts[i]}'")
        print(f"   '{test_texts[j]}'")
        print(f"   相似度: {sim:.4f} {'⭐ 高相似度!' if sim > 0.5 else ''}")



---
## 🗄️ Part 5: 向量存儲 (Vector Store)

Vector Store 用於存儲和高效檢索向量。我們使用 ChromaDB 作為本地向量資料庫。

In [ ]:
from langchain_community.vectorstores import Chroma

# 建立向量存儲 (將所有 chunks 轉換為向量並存儲)
print("⏳ 正在建立向量資料庫...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",  # 持久化存儲位置
    collection_name="alice_wonderland"
)

print(f"✅ 向量資料庫建立完成！")
print(f"📊 共存儲 {vectorstore._collection.count()} 個向量")



In [ ]:
# 測試基本檢索
query = "What happened when Alice fell down the rabbit hole?"

# 執行相似度搜索
results = vectorstore.similarity_search(query, k=3)

print(f"🔍 查詢: '{query}'")
print("=" * 60)

for i, doc in enumerate(results):
    print(f"\n📄 結果 {i+1}:")
    print(f"   來源位置: {doc.metadata.get('start_index', 'N/A')}")
    print(f"   內容: {doc.page_content[:250]}...")
    print("-" * 60)

Loaded 1 document(s).
First 200 characters of the document:
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Ho


---
## 🤖 Part 6: 完整 RAG Pipeline

現在讓我們整合所有組件，建立完整的 RAG 問答系統！



In [ ]:
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

# 建立 Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# 建立本地 LLM
llm = OllamaLLM(
    model=MODEL_NAME,
    temperature=0.7
)

print("✅ Retriever 和 LLM 已準備就緒！")

: 

In [ ]:
# 自定義 Prompt Template
template = """你是一個專業的閱讀助手，請根據提供的上下文回答問題。
如果上下文中沒有相關資訊，請誠實說明你無法從提供的資料中找到答案。

上下文:
{context}

問題: {question}

請用繁體中文回答:"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

print("✅ Prompt Template 已建立！")



In [ ]:
# 建立 RAG Chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # 將所有檢索到的文件合併
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

print("✅ RAG Chain 已建立完成！")

: 

In [ ]:
# 測試 RAG 系統
def ask_question(question):
    print(f"\n❓ 問題: {question}")
    print("=" * 60)

    result = rag_chain.invoke({"query": question})

    print(f"\n💡 回答:")
    print(result["result"])

    print(f"\n📚 參考來源:")
    for i, doc in enumerate(result["source_documents"]):
        print(f"   {i+1}. 位置 {doc.metadata.get('start_index', 'N/A')}: {doc.page_content[:80]}...")

    return result

# 測試問題
result = ask_question("愛麗絲是如何進入仙境的？")



In [ ]:
# 更多測試問題
test_questions = [
    "愛麗絲在茶會上遇到了誰？",
    "紅心皇后是什麼樣的人物？",
    "柴郡貓有什麼特別的能力？",
]

for q in test_questions:
    ask_question(q)
    print("\n" + "="*80 + "\n")

Installed langchain-text-splitters.


---
## 🎯 Lab 01 總結

在本 Lab 中，我們學習了：

1. **文件加載**: 使用 `TextLoader` 載入文本文件
2. **文本分割**: 使用 `RecursiveCharacterTextSplitter` 將長文件分割為小 chunks
3. **向量嵌入**: 使用本地 `HuggingFaceEmbeddings` 模型生成向量
4. **向量存儲**: 使用 `ChromaDB` 存儲和檢索向量
5. **RAG Pipeline**: 整合 `Ollama` 本地 LLM 建立完整問答系統

### 下一步
- **Lab 02**: 深入探討不同 Vector Database 的特性與操作
- **Lab 03**: 學習 Hybrid Search 混合搜索策略
- **Lab 04**: 進階 RAG 技術 - Query Rewriting & Reranking
- **Lab 05**: 使用 RAGAS 評估 RAG 系統品質



In [ ]:
# 清理資源 (可選)
# import shutil
# shutil.rmtree("./chroma_db", ignore_errors=True)
# print("✅ 已清理向量資料庫")

print("🎉 Lab 01 完成！")
print(f"First chunk metadata:\n{chunks[0].metadata}")

Split 1 document(s) into 191 chunks.
First chunk (first 200 chars):
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Ho
First chunk metadata:
{'source': 'data/alice_in_wonderland.txt', 'start_index': 0}


**Reasoning**:
Following the instructions, the next step is to initialize an embedding model using `SentenceTransformerEmbeddings` from `langchain_community.embeddings` and generate embeddings for the document chunks. The model 'all-MiniLM-L6-v2' is specified for this purpose.



In [13]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

# Initialize the embedding model
embeddings_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

print("Embedding model initialized successfully.")

/tmp/ipython-input-1724011893.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model initialized successfully.


**Reasoning**:
The embedding model has been successfully initialized. The next step, as per the instructions, is to create a vector store (e.g., using `Chroma` from `langchain_community.vectorstores`) from the generated embeddings and document chunks, and then perform a similarity search using a sample query.



In [14]:
from langchain_community.vectorstores import Chroma

# Create a vector store from the chunks and embeddings
# This will embed the chunks and store them in a ChromaDB instance
vectorstore = Chroma.from_documents(chunks, embeddings_model)

print(f"Vector store created with {vectorstore._collection.count()} items.")

# Perform a similarity search (retrieval)
query = "What is Alice's adventure about?"
docs = vectorstore.similarity_search(query)

print(f"\nRetrieved {len(docs)} documents for the query: '{query}'")
print("--------------------------------------------------")
for i, doc in enumerate(docs):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}, Start Index: {doc.metadata.get('start_index')}):\n{doc.page_content[:500]}...\n")

Vector store created with 191 items.

Retrieved 4 documents for the query: 'What is Alice's adventure about?'
--------------------------------------------------
Document 1 (Source: data/alice_in_wonderland.txt, Start Index: 83988):
“At any rate I’ll never go _there_ again!” said Alice as she picked her
way through the wood. “It’s the stupidest tea-party I ever was at in
all my life!”

Just as she said this, she noticed that one of the trees had a door
leading right into it. “That’s very curious!” she thought. “But
everything’s curious today. I think I may as well go in at once.” And
in she went.

Once more she found herself in the long hall, and close to the little
glass table. “Now, I’ll manage better this time,” she said...

Document 2 (Source: data/alice_in_wonderland.txt, Start Index: 79779):
“Tell us a story!” said the March Hare.

“Yes, please do!” pleaded Alice.

“And be quick about it,” added the Hatter, “or you’ll be asleep again
before it’s done.”

“Once upon a time there wer

## Generate Lab 2: Vector DB Deep Dive

### Subtask:
Provide the Markdown and Code cells for resource estimation and metadata filtering in Vector Databases.


## Lab 2: Vector DB Deep Dive

This lab delves deeper into the practical aspects of working with Vector Databases, focusing on crucial considerations for building robust and efficient RAG systems. Understanding these concepts is vital for optimizing performance and managing resources effectively.

### Topics Covered:
1.  **Resource Estimation**: How to estimate the memory requirements for storing embeddings in a vector database, based on factors like the number of chunks and embedding dimensions.
2.  **Metadata Filtering**: Enhancing retrieval accuracy and efficiency by leveraging metadata to narrow down search results, allowing for more precise contextual retrieval.

### Resource Estimation for Vector Databases

When building RAG systems, it's crucial to understand the resource implications of storing embeddings. Vector databases store high-dimensional numerical representations of text, and their memory footprint can grow significantly with the volume of data.

Key factors influencing resource estimation:
-   **Number of Documents/Chunks**: Each chunk of text will be converted into a vector. More chunks mean more vectors to store.
-   **Embedding Dimension**: The size of each vector (the number of dimensions) directly impacts memory usage. Larger dimensions lead to larger vectors.
-   **Data Type**: Embeddings are typically stored as floating-point numbers. `float32` (4 bytes per number) is common, but `float16` (2 bytes) can halve memory usage at the cost of some precision, and `float64` (8 bytes) offers higher precision but doubles memory.

To estimate the memory required, you can use the formula: `Number of Chunks * Embedding Dimension * Bytes per float`.

**Reasoning**:
Following the instructions, the next step is to create a Code cell to calculate the estimated memory usage for the existing vector store, using the number of chunks and the embedding dimension with a float32 assumption.



In [15]:
import sys

# Get the number of chunks
num_chunks = len(chunks)

# Determine the embedding dimension
# We'll embed a sample query and get the length of the resulting vector
sample_embedding = embeddings_model.embed_query('test')
embedding_dimension = len(sample_embedding)

# Assume float32 for embedding storage (4 bytes per dimension)
bytes_per_float = 4

# Calculate total estimated memory
estimated_memory_bytes = num_chunks * embedding_dimension * bytes_per_float
estimated_memory_mb = estimated_memory_bytes / (1024**2)
estimated_memory_gb = estimated_memory_bytes / (1024**3)

print(f"Number of chunks: {num_chunks}")
print(f"Embedding dimension: {embedding_dimension}")
print(f"Assumed bytes per float: {bytes_per_float}")
print(f"Estimated memory for embeddings: {estimated_memory_bytes:.2f} bytes")
print(f"Estimated memory for embeddings: {estimated_memory_mb:.2f} MB")
print(f"Estimated memory for embeddings: {estimated_memory_gb:.2f} GB")

Number of chunks: 191
Embedding dimension: 384
Assumed bytes per float: 4
Estimated memory for embeddings: 293376.00 bytes
Estimated memory for embeddings: 0.28 MB
Estimated memory for embeddings: 0.00 GB


### Metadata Filtering

Metadata filtering is a powerful technique to enhance the precision and relevance of retrieval in RAG systems. While vector similarity search is excellent for semantic matching, it doesn't always capture all the nuances or specific requirements of a query.

Metadata refers to additional information or attributes associated with each document or chunk. This can include:
-   **Categorical data**: document type (e.g., 'Introduction', 'Main Body', 'Appendix'), author, publication year, topic.
-   **Numerical data**: `start_index` (position in the original document), page number, section ID.

By leveraging metadata filters, you can:
-   **Narrow down search space**: Restrict similarity search to only chunks that match specific criteria, significantly improving efficiency and relevance.
-   **Improve precision**: Ensure retrieved documents meet specific structural or contextual requirements (e.g., "find answers about Alice's adventures only from the 'Main Body' section").
-   **Enhance control**: Provide more granular control over what information is presented to the language model.

Vector databases like Chroma support robust metadata filtering, allowing you to combine semantic similarity with structured filtering conditions.

**Reasoning**:
Following the instructions, the next step is to create a Code cell to demonstrate metadata filtering. This involves preparing chunks with new metadata, re-creating the vector store, and then performing both numerical and categorical filtered retrievals.



In [16]:
from langchain_core.documents import Document

# a. Prepare Chunks with Metadata
updated_chunks = []
for i, chunk in enumerate(chunks):
    new_metadata = chunk.metadata.copy()
    if i < 10:
        new_metadata['doc_part'] = 'Introduction'
    elif i < 100:
        new_metadata['doc_part'] = 'Main Body'
    else:
        new_metadata['doc_part'] = 'Appendix'
    updated_chunks.append(Document(page_content=chunk.page_content, metadata=new_metadata))

print(f"Prepared {len(updated_chunks)} chunks with updated metadata.")
print(f"Example of updated metadata for first chunk: {updated_chunks[0].metadata}")
print(f"Example of updated metadata for chunk 15: {updated_chunks[15].metadata}")
print(f"Example of updated metadata for chunk 105: {updated_chunks[105].metadata}")

# b. Re-create Vector Store with updated chunks and embeddings
# We'll use a new Chroma instance to demonstrate the filtering clearly
vectorstore_filtered = Chroma.from_documents(updated_chunks, embeddings_model, collection_name="alice_filtered_data")

print(f"Vector store with filtered metadata created with {vectorstore_filtered._collection.count()} items.")

# c. Perform Filtered Retrieval (Numerical Metadata)
query_num = "Who is the main character and what is her story about?"
# Filter for chunks that start after index 50000
docs_num_filtered = vectorstore_filtered.similarity_search(query_num, k=3, where={"start_index": {"$gt": 50000}})

print(f"\nRetrieved {len(docs_num_filtered)} documents for query '{query_num}' with numerical filter (start_index > 50000):")
print("--------------------------------------------------")
for i, doc in enumerate(docs_num_filtered):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}, Start Index: {doc.metadata.get('start_index')}, Doc Part: {doc.metadata.get('doc_part')}):\n{doc.page_content[:200]}...\n")

# d. Perform Filtered Retrieval (Categorical Metadata)
query_cat = "Who does Alice meet at a tea party?"
# Filter for chunks only from 'Main Body'
docs_cat_filtered = vectorstore_filtered.similarity_search(query_cat, k=3, where={"doc_part": "Main Body"})

print(f"\nRetrieved {len(docs_cat_filtered)} documents for query '{query_cat}' with categorical filter (doc_part = 'Main Body'):")
print("--------------------------------------------------")
for i, doc in enumerate(docs_cat_filtered):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}, Start Index: {doc.metadata.get('start_index')}, Doc Part: {doc.metadata.get('doc_part')}):\n{doc.page_content[:200]}...\n")

: 